In [1]:
# 1. Establish API key

import os

#GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

#ANTHROPIC_API_KEY

if OPENAI_API_KEY is None:
    raise RuntimeError("OPENAI_API_KEY not found in environment.")

In [2]:
# 2. Set automation constants
import os, json

FEATURES = [
    "AktAcc", "AktAch", "AktAct", "AktSemel", "AktState",
    "ASTrns", "ASUncc", "ASUnrg", "ClCntfcl", "ClNegPol",
    "DtDTAM", "DtExp", "DtRes", "DtUniv",
    "TLAspPfv", "TLMdIrr", "TLTS", "VcPass",
]

PERMUTATIONS = [
    {"type": "full",     "component": None},
    {"type": "ablation", "component": "intuition"},
    {"type": "ablation", "component": "set_to_1_if"},
    {"type": "ablation", "component": "set_to_0_if"},
    {"type": "ablation", "component": "edge_cases"},
    {"type": "ablation", "component": "examples_positive"},
    {"type": "ablation", "component": "examples_negative"},
    {"type": "allation", "component": "intuition"},
    {"type": "allation", "component": "set_to_1_if"},
    {"type": "allation", "component": "set_to_0_if"},
    {"type": "allation", "component": "edge_cases"},
    {"type": "allation", "component": "examples_positive"},
    {"type": "allation", "component": "examples_negative"},
]

BASE_ROOT  = "/Users/glossophilia/Desktop/cri_runs/OpenAI"
CRI_DIR    = "/Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/p1_cri_pipeline/cris"
INPUT_CSV  = "/Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/p1_cri_pipeline/input/ntb_input.csv"
CHECKPOINT = os.path.join(BASE_ROOT, "completed_runs.json")

In [3]:
# 3. Run config specs

run_config = {
  "run_id": None,

  #"feature_name": "DtUniv",  # <-- set this

  "model_provider": "openai",  # <-- set this
  "model_name": "gpt-4.1",  # <-- set this
    # Set input_csv_path to match data input file and cri_json_path to feature-specific CRI file
  "input_csv_path": "/Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/p1_cri_pipeline/input/ntb_input.csv",
  #"cri_json_path": "/Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/p1_cri_pipeline/cris/DtUniv.json",

  "data": {
      "max_tokens": 350,    # None = all; total tokens to draw from dataset
      "batch_size": 50,     # tokens per output file
      "start_index": 0,
      "shuffle": False,
      "random_seed": 0
  },

 # "permutation": {
 #   "type": "ablation",            # full | ablation | allation
 #   "component": "intuition"
 # },

  "prompt": {
    "version": "v1",
    "system_template_hash": None,
    "assembled_prompt_hash": None
  },

  "inference": {
    "temperature": 0.0,
    "max_tokens": 512,
    "seed": None,
    "rate_limit_sleep_s": 1.0,
    "max_retries": 5
  },

  "io": {
    "output_dir": None,
    "jsonl_output_path": None,
    "csv_output_path": None
  },

  "notes": ""
}

def _derive_cri_variant(permutation: dict) -> str:
    abbrev   = {"full": "full", "ablation": "abl", "allation": "all"}
    comp_idx = {
        "intuition": 1, "set_to_1_if": 2, "set_to_0_if": 3,
        "edge_cases": 4, "examples_positive": 5, "examples_negative": 6,
    }
    t = abbrev.get(permutation["type"], permutation["type"])
    c = permutation.get("component")
    return t if c is None else f"{t}{comp_idx.get(c, c)}"

#run_config["cri_variant"] = _derive_cri_variant(run_config["permutation"])
#print("cri_variant:", run_config["cri_variant"])

In [4]:
# 3. Set checkpoints

def _ckpt_key(feature_name, permutation):
    return f"{feature_name}::{_derive_cri_variant(permutation)}"

def load_checkpoint():
    if os.path.exists(CHECKPOINT):
        with open(CHECKPOINT) as f:
            return set(json.load(f))
    return set()

def save_checkpoint(completed):
    with open(CHECKPOINT, "w") as f:
        json.dump(sorted(completed), f, indent=2)

In [5]:
# 4. Load CRI json file and implement permutation (full/ablation/allation)

import json
from copy import deepcopy

CRI_COMPONENTS = [
    "intuition",
    "set_to_1_if",
    "set_to_0_if",
    "edge_cases",
    "examples_positive",
    "examples_negative",
]

def load_cri(cri_json_path: str) -> dict:
    with open(cri_json_path, "r", encoding="utf-8") as f:
        cri = json.load(f)
    return cri

def apply_permutation(cri: dict, permutation: dict) -> dict:
    """
    Returns a CRI dict containing ONLY the active components for this run.
    - full: all components kept
    - ablation: drop exactly one named component
    - allation: keep exactly one named component
    """
    ptype = permutation["type"]
    component = permutation.get("component")

    # Defensive copy
    cri_active = deepcopy(cri)

    if ptype == "full":
        # Ensure we only keep the known components (avoids stray metadata leaking into prompt)
        return {k: cri_active.get(k) for k in CRI_COMPONENTS if k in cri_active}

    if component not in CRI_COMPONENTS:
        raise ValueError(f"Invalid component '{component}'. Must be one of: {CRI_COMPONENTS}")

    if ptype == "ablation":
        return {k: cri_active.get(k) for k in CRI_COMPONENTS if k != component and k in cri_active}

    if ptype == "allation":
        return {component: cri_active.get(component)}

    raise ValueError(f"Invalid permutation type '{ptype}'. Must be: full | ablation | allation")

# ---- Fill these in for your current run ----
# Set desired output folder path
#run_config["cri_json_path"] = "/Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/p1_cri_pipeline/cris/DtExp.json"   # <-- set this
# run_config["permutation"] = {"type": "ablation", "component": "edge_cases"}  # example
# run_config["permutation"] = {"type": "allation", "component": "set_to_1_if"} # example

#cri = load_cri(run_config["cri_json_path"])
#cri_active = apply_permutation(cri, run_config["permutation"])

#print("Active CRI components:", list(cri_active.keys()))

In [6]:
# 5. Assemble a deterministic prompt string + write its hash into run_config

import hashlib

def sha256_text(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

PROMPT_TEMPLATE_V1 = """\
You are a trained linguist performing binary semantic feature annotation.

TASK:
Decide whether the TARGET GRAM instantiates the feature: {feature_name}.

EVIDENCE RULES:
- Use only the TARGET GRAM and the FULL TEXT provided.
- Follow the Constrained Reasoning Instructions (CRIs) exactly as written.
- Do not assume missing context or world knowledge beyond what is in the FULL TEXT.

META-INSTRUCTIONS (how to use the CRIs):
{meta_block}

CRIs:
{cri_block}

INPUT:
TokenID: {TokenID}
Target: {Target}
FullText: {FullText}

OUTPUT (STRICT JSON ONLY; no extra text):
{{
  "TokenID": "{TokenID}",
  "annotation": 0 or 1,
  "confidence": "high" or "medium" or "low",
  "explanation": "one sentence"
}}
"""

def render_meta_block(cri_active: dict) -> str:
    lines = []

    # Always present
    lines.append("Treat each CRI component as a different kind of evidence. Follow them as written.")

    # Only if set_to_1_if exists
    if "set_to_1_if" in cri_active:
        lines.append("Assign 1 if at least one 'set_to_1_if' condition applies.")

    # Only if set_to_0_if exists
    if "set_to_0_if" in cri_active:
        lines.append(
            "If any 'set_to_0_if' condition applies, assign 0 (this vetoes 'set_to_1_if'), "
            "unless an 'edge_cases' instruction explicitly overrides."
        )

    return "- " + "\n- ".join(lines)

def render_cri_block(cri_active: dict) -> str:
    parts = []
    for key in CRI_COMPONENTS:
        if key in cri_active and cri_active[key] is not None:
            parts.append(f"[{key}]\n{cri_active[key]}")
    return "\n\n".join(parts)

def build_prompt(feature_name: str, cri_active: dict, token: dict) -> str:
    cri_block = render_cri_block(cri_active)
    meta_block = render_meta_block(cri_active)

    return PROMPT_TEMPLATE_V1.format(
        feature_name=feature_name,
        cri_block=cri_block,
        meta_block=meta_block,
        TokenID=token["TokenID"],
        Target=token["Target"],
        FullText=token["FullText"],
    )

# ---- store template hash in config ----
run_config["prompt"]["version"] = "v1"
run_config["prompt"]["system_template_hash"] = sha256_text(PROMPT_TEMPLATE_V1)

print("Template hash:", run_config["prompt"]["system_template_hash"])

Template hash: a162a52f2df71cfe965f20df69fa760eb7aaf1f55cd929401825f456846ab44a


In [7]:
# 6. Load input CSV, apply slice, set output filenames, and render one dry-run prompt

import pandas as pd
import os
import json
from datetime import datetime

# ---- Load CSV once ----
df = pd.read_csv(run_config["input_csv_path"])

required_columns = {"TokenID", "Target", "FullText"}
missing = required_columns - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# ---- Apply run_config data slicing deterministically ----
data_cfg = run_config["data"]

df_work = df
if data_cfg.get("shuffle", False):
    df_work = df_work.sample(frac=1, random_state=data_cfg.get("random_seed", 0)).reset_index(drop=True)

start = int(data_cfg.get("start_index", 0))
max_tokens = data_cfg.get("max_tokens", None)

df_slice = df_work.iloc[start:] if max_tokens is None else df_work.iloc[start:start + int(max_tokens)]
tokens = df_slice.to_dict(orient="records")

print(f"Total tokens in file: {len(df)}")
print(f"Tokens selected for this run: {len(tokens)}")

if not tokens:
    raise RuntimeError("No tokens selected for this run (tokens list is empty).")

Total tokens in file: 349
Tokens selected for this run: 349


In [8]:
# 7. Strict response parser + validator

import json
import re

_FENCE_RE = re.compile(r"^\s*```(?:json)?\s*\n([\s\S]*?)\n```\s*$", re.IGNORECASE)

def parse_model_response(raw_text: str) -> dict:
    """
    Parses model output and enforces strict JSON schema.
    Accepts either:
      - pure JSON
      - JSON wrapped in a single ```json ... ``` fence
    """
    text = raw_text.strip()

    m = _FENCE_RE.match(text)
    if m:
        text = m.group(1).strip()

    try:
        data = json.loads(text)
    except json.JSONDecodeError as e:
        raise ValueError(f"Model output is not valid JSON. JSONDecodeError: {e}")

    required_keys = {"TokenID", "annotation", "confidence", "explanation"}
    if set(data.keys()) != required_keys:
        raise ValueError(f"JSON keys mismatch. Expected {required_keys}, got {set(data.keys())}")

    if data["annotation"] not in (0, 1):
        raise ValueError("Annotation must be 0 or 1.")

    if data["confidence"] not in ("high", "medium", "low"):
        raise ValueError("Confidence must be 'high', 'medium', or 'low'.")

    if not isinstance(data["TokenID"], str):
        raise ValueError("TokenID must be a string.")

    if not isinstance(data["explanation"], str):
        raise ValueError("Explanation must be a string.")

    return data

In [9]:
# 8. Model call wrapper (OpenAI + provider dispatcher)

import time
import openai
from openai import OpenAI, RateLimitError, APIError

# ---- OpenAI client ----
client = OpenAI(api_key=OPENAI_API_KEY)

def call_openai(prompt: str) -> str:
    max_retries = int(run_config["inference"].get("max_retries", 3))
    base_sleep  = float(run_config["inference"].get("rate_limit_sleep_s", 1.0))
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            response = client.chat.completions.create(
                model=run_config["model_name"],
                messages=[{"role": "user", "content": prompt}],
                temperature=run_config["inference"]["temperature"],
                max_completion_tokens=int(run_config["inference"]["max_tokens"]),
            )
            return response.choices[0].message.content
        except RateLimitError as e:
            last_err = e
            sleep_s = max(base_sleep, 1.0) * (2 ** (attempt - 1))
            print(f"[429] Rate limited. Sleeping {sleep_s:.1f}s then retrying ({attempt}/{max_retries})...")
            time.sleep(sleep_s)
            continue
        except APIError as e:
            raise
    raise RuntimeError(f"OpenAI call failed after {max_retries} retries. Last error: {last_err}")

# ---- Provider dispatcher ----
def call_model(prompt: str) -> str:
    provider = run_config["model_provider"]
    if provider == "openai":
        return call_openai(prompt)
    else:
        raise ValueError(f"Unknown model_provider: {provider}")

In [11]:
# Run ONE test call and inspect token usage returned by OpenAI.

from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)

import json

cri_path = "/Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/p1_cri_pipeline/cris/AktState.json"
cri_active = json.load(open(cri_path, "r", encoding="utf-8"))

# Build ONE real prompt the same way you do in the pipeline.
# (Use an existing token row + the CRI you used.)
prompt = build_prompt(
    feature_name="AktState",      # or whatever you're testing right now
    cri_active=cri_active,        # whatever dict you're passing into build_prompt
    token=tokens[0],              # pick one real token dict from your loaded dataset
)

resp = client.chat.completions.create(
    model=run_config["model_name"],   # e.g., "gpt-4.1"
    messages=[{"role": "user", "content": prompt}],
    temperature=run_config["inference"]["temperature"],
    max_completion_tokens=run_config["inference"]["max_tokens"],  # IMPORTANT param name
)

print("USAGE:", getattr(resp, "usage", None))
print("TEXT:", resp.choices[0].message.content[:200])
print("PROMPT CHARS:", len(prompt))

USAGE: CompletionUsage(completion_tokens=69, prompt_tokens=759, total_tokens=828, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))
TEXT: {
  "TokenID": "40001018",
  "annotation": 0,
  "confidence": "high",
  "explanation": "The target 'had been promised' is in the progressive perfect passive form, which is morphologically marked as pr
PROMPT CHARS: 3175
